# Task-Aware Enhancement Model for Glaucoma Detection

This notebook demonstrates the **Enhancement Model (M_E)**, which is specifically trained to optimize the segmentation performance for Optic Disc (OD) and Optic Cup (OC).

## Architecture: Hybrid U-Net with Residual Blocks

- **U-Net Backbone**: Gold standard for image-to-image reconstruction
- **Residual Blocks**: Bottleneck design from CycleGAN for efficiency
- **Leaky ReLU**: Preserves information at low pixel values
- **Task-Aware Loss**: Directly optimizes segmentation performance

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

## 2. Enhancement U-Net Model

The Enhancement Model consists of:
- **Encoder**: 4 downsampling stages with skip connections
- **Bottleneck**: 6 Residual Blocks for efficient feature processing
- **Decoder**: 4 upsampling stages with skip connections
- **Output**: Enhanced RGB image [0, 1]

In [ ]:
from src.models.enhancement_unet import (
    EnhancementUNet,
    EnhancementSegmentationPipeline,
    ResidualBlock
)
from src.models.unet import UNet

# Initialize Enhancement Model
enhancement_model = EnhancementUNet(
    n_channels=3,
    n_output_channels=3,
    base_channels=64,
    n_residual_blocks=6
)

print("Enhancement Model Architecture:")
print("=" * 80)
print(enhancement_model)
print("=" * 80)
print(f"Total Parameters: {sum(p.numel() for p in enhancement_model.parameters()):,}")
print(f"Trainable Parameters: {sum(p.numel() for p in enhancement_model.parameters() if p.requires_grad):,}")

## 3. Test the Enhancement Model

Let's test the model with a random input image:

In [ ]:
# Test with random input
device = 'cuda' if torch.cuda.is_available() else 'cpu'
enhancement_model = enhancement_model.to(device)
enhancement_model.eval()

# Create test input
batch_size = 2
test_input = torch.randn(batch_size, 3, 512, 512).to(device)

# Forward pass
with torch.no_grad():
    enhanced_output = enhancement_model(test_input)

print(f"Input Shape: {test_input.shape}")
print(f"Output Shape: {enhanced_output.shape}")
print(f"Input Range: [{test_input.min():.3f}, {test_input.max():.3f}]")
print(f"Output Range: [{enhanced_output.min():.3f}, {enhanced_output.max():.3f}]")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normalize input for visualization
input_vis = test_input[0].cpu().permute(1, 2, 0).numpy()
input_vis = (input_vis - input_vis.min()) / (input_vis.max() - input_vis.min())

output_vis = enhanced_output[0].cpu().permute(1, 2, 0).numpy()

axes[0].imshow(input_vis)
axes[0].set_title('Input (normalized)')
axes[0].axis('off')

axes[1].imshow(output_vis)
axes[1].set_title('Enhanced Output')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Load Segmentation Model

The segmentation model is frozen and only used to compute the task-aware loss:

In [ ]:
# Initialize Segmentation Model
segmentation_model = UNet(
    n_channels=3,
    n_classes=3,  # Background, OD, OC
    base_channels=64
)

# Optional: Load pre-trained model
checkpoint_path = project_root / 'experiments' / 'segmentation' / 'best_model.pth'

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        segmentation_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        segmentation_model.load_state_dict(checkpoint)
    print(f"✓ Loaded pre-trained segmentation model from {checkpoint_path}")
else:
    print("⚠ No pre-trained model found, using randomly initialized weights")

segmentation_model = segmentation_model.to(device)
print(f"Segmentation Model Parameters: {sum(p.numel() for p in segmentation_model.parameters()):,}")

## 5. Create End-to-End Pipeline

The pipeline combines Enhancement and Segmentation:

```
Input → Enhancement (M_E) → Enhanced → Segmentation (M_S) → Output
  ↓                          ↓                              ↓
  └── Perceptual Loss ───────┘                   Dice Loss ─┘
```

In [ ]:
# Create pipeline
pipeline = EnhancementSegmentationPipeline(
    enhancement_model=enhancement_model,
    segmentation_model=segmentation_model,
    freeze_segmentation=True  # Freeze segmentation
)

pipeline = pipeline.to(device)
pipeline.eval()

print("End-to-End Pipeline:")
print(f"  Total Parameters: {sum(p.numel() for p in pipeline.parameters()):,}")
print(f"  Trainable Parameters: {sum(p.numel() for p in pipeline.parameters() if p.requires_grad):,}")
print(f"  Segmentation Frozen: {not any(p.requires_grad for p in segmentation_model.parameters())}")
print(f"  Enhancement Trainable: {all(p.requires_grad for p in enhancement_model.parameters())}")

## 6. Test the Pipeline

Let's test the complete pipeline:

In [ ]:
# Test Pipeline
test_input = torch.randn(2, 3, 512, 512).to(device)

with torch.no_grad():
    enhanced, segmentation = pipeline(test_input)

print(f"Input Shape: {test_input.shape}")
print(f"Enhanced Shape: {enhanced.shape}")
print(f"Segmentation Shape: {segmentation.shape}")

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Input
input_vis = test_input[0].cpu().permute(1, 2, 0).numpy()
input_vis = (input_vis - input_vis.min()) / (input_vis.max() - input_vis.min())
axes[0].imshow(input_vis)
axes[0].set_title('Input')
axes[0].axis('off')

# Enhanced
enhanced_vis = enhanced[0].cpu().permute(1, 2, 0).numpy()
axes[1].imshow(enhanced_vis)
axes[1].set_title('Enhanced')
axes[1].axis('off')

# Segmentation (argmax)
seg_pred = torch.argmax(segmentation[0], dim=0).cpu().numpy()
axes[2].imshow(seg_pred, cmap='jet', vmin=0, vmax=2)
axes[2].set_title('Segmentation')
axes[2].axis('off')

# Segmentation probabilities
seg_prob = torch.softmax(segmentation[0], dim=0)[1].cpu().numpy()  # OD probability
axes[3].imshow(seg_prob, cmap='hot')
axes[3].set_title('OD Probability')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 7. Task-Aware Loss Functions

The loss function combines:
- **Dice Loss**: Optimizes segmentation performance (Task-Aware)
- **Perceptual Loss**: Preserves image quality

$$\mathcal{L}_{\text{total}} = \lambda_{\text{dice}} \cdot \mathcal{L}_{\text{dice}} + \lambda_{\text{perceptual}} \cdot \mathcal{L}_{\text{perceptual}}$$

In [ ]:
from src.training.task_aware_train import (
    TaskAwareLoss,
    DiceLoss,
    PerceptualLoss,
    compute_dice_score
)

# Create loss function
criterion = TaskAwareLoss(
    lambda_dice=1.0,         # Weight for Task-Aware Loss
    lambda_perceptual=0.1,   # Weight for Quality Loss
    n_classes=3
)

print("Task-Aware Loss Function:")
print(f"  λ_dice = {criterion.lambda_dice}")
print(f"  λ_perceptual = {criterion.lambda_perceptual}")

## 8. Test Loss Functions

In [ ]:
# Create dummy data for loss test
batch_size = 4
height, width = 256, 256

dummy_logits = torch.randn(batch_size, 3, height, width).to(device)
dummy_targets = torch.randint(0, 3, (batch_size, height, width)).to(device)
dummy_enhanced = torch.randn(batch_size, 3, height, width).to(device)
dummy_original = torch.randn(batch_size, 3, height, width).to(device)

# Compute loss
loss, loss_dict = criterion(
    segmentation_logits=dummy_logits,
    targets=dummy_targets,
    enhanced_images=dummy_enhanced,
    original_images=dummy_original
)

print("Loss Values:")
print(f"  Total Loss: {loss_dict['total']:.4f}")
print(f"  Dice Loss: {loss_dict['dice']:.4f}")
print(f"  Perceptual Loss: {loss_dict['perceptual']:.4f}")

# Compute metrics
metrics = compute_dice_score(dummy_logits, dummy_targets, n_classes=3)

print("\nMetrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 9. Load Dataset

In [ ]:
from src.data_loader.dataset import GlaucomaDataset
from src.data_loader.transforms import get_training_transforms, get_validation_transforms

# Create transforms
train_transform = get_training_transforms(image_size=512)
val_transform = get_validation_transforms(image_size=512)

# Create datasets
train_dataset = GlaucomaDataset(
    root_dir=str(project_root),
    split='train',
    transform=train_transform
)

val_dataset = GlaucomaDataset(
    root_dir=str(project_root),
    split='val',
    transform=val_transform
)

print(f"Training Samples: {len(train_dataset)}")
print(f"Validation Samples: {len(val_dataset)}")

# Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

## 10. Visualize Sample Data

In [ ]:
# Load a batch
sample_images, sample_masks = next(iter(train_loader))

print(f"Batch Images Shape: {sample_images.shape}")
print(f"Batch Masks Shape: {sample_masks.shape}")
print(f"Image Range: [{sample_images.min():.3f}, {sample_images.max():.3f}]")
print(f"Mask Values: {torch.unique(sample_masks)}")

# Visualize first 4 samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(4, len(sample_images))):
    # Image
    img = sample_images[i].permute(1, 2, 0).numpy()
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    
    # Mask
    mask = sample_masks[i].numpy()
    axes[1, i].imshow(mask, cmap='jet', vmin=0, vmax=2)
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 11. Training Setup

Let's configure the training:

In [ ]:
from src.training.task_aware_train import EnhancementTrainer

# Optimizer (only for Enhancement Model, since Segmentation is frozen)
optimizer = optim.Adam(
    enhancement_model.parameters(),
    lr=0.0002,
    betas=(0.5, 0.999)
)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    verbose=True
)

# Create trainer
trainer = EnhancementTrainer(
    pipeline=pipeline,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler
)

print("Training Setup Complete!")
print(f"  Device: {device}")
print(f"  Optimizer: Adam (lr={optimizer.param_groups[0]['lr']})")
print(f"  Scheduler: ReduceLROnPlateau")

## 12. Mini-Training Loop (Demo)

Let's run a few training steps to show how training works:

In [ ]:
# Mini-Training for demo (only 5 batches)
num_batches = 5
pipeline.train()

train_losses = []
train_dice_scores = []

print("Running Mini-Training Demo...")
print("=" * 80)

for batch_idx, (images, masks) in enumerate(train_loader):
    if batch_idx >= num_batches:
        break
    
    # Training step
    loss_dict, metrics_dict = trainer.train_step(images, masks)
    
    train_losses.append(loss_dict['total'])
    train_dice_scores.append(metrics_dict['dice_mean'])
    
    print(f"Batch {batch_idx+1}/{num_batches}:")
    print(f"  Loss: {loss_dict['total']:.4f} (Dice: {loss_dict['dice']:.4f}, Perceptual: {loss_dict['perceptual']:.4f})")
    print(f"  Dice OD: {metrics_dict['dice_od']:.4f}, Dice OC: {metrics_dict['dice_oc']:.4f}, Mean: {metrics_dict['dice_mean']:.4f}")
    print()

print("=" * 80)
print(f"Average Loss: {np.mean(train_losses):.4f}")
print(f"Average Dice: {np.mean(train_dice_scores):.4f}")

## 13. Visualize Enhancement Results

Let's see how the Enhancement Model improves the images:

In [ ]:
# Validation on one batch
pipeline.eval()
val_images, val_masks = next(iter(val_loader))

with torch.no_grad():
    val_images = val_images.to(device)
    val_masks = val_masks.to(device)
    
    enhanced, segmentation = pipeline(val_images)
    seg_pred = torch.argmax(segmentation, dim=1)

# Visualize first 2 samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(min(2, len(val_images))):
    # Original
    img = val_images[i].cpu().permute(1, 2, 0).numpy()
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Original Image')
    axes[i, 0].axis('off')
    
    # Enhanced
    enh = enhanced[i].cpu().permute(1, 2, 0).numpy()
    axes[i, 1].imshow(enh)
    axes[i, 1].set_title('Enhanced Image')
    axes[i, 1].axis('off')
    
    # Ground Truth Mask
    mask = val_masks[i].cpu().numpy()
    axes[i, 2].imshow(mask, cmap='jet', vmin=0, vmax=2)
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')
    
    # Predicted Mask
    pred = seg_pred[i].cpu().numpy()
    axes[i, 3].imshow(pred, cmap='jet', vmin=0, vmax=2)
    axes[i, 3].set_title('Prediction')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

# Berechne Metriken
metrics = compute_dice_score(segmentation, val_masks, n_classes=3)
print("\nValidation Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 14. Vollständiges Training starten

Um das vollständige Training zu starten, verwenden Sie das Trainingsskript:

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║                                                                            ║
║                    Vollständiges Training starten                          ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝

Um das vollständige Training zu starten, öffnen Sie ein Terminal und führen Sie aus:

1. Mit Konfigurationsdatei:
   
   source .venv/bin/activate && python src/training/train_enhancement.py \\
       --config configs/enhancement_config.yaml

2. Mit benutzerdefinierten Parametern:
   
   source .venv/bin/activate && python src/training/train_enhancement.py \\
       --epochs 100 \\
       --batch_size 8 \\
       --lr 0.0002 \\
       --enhancement_channels 64 \\
       --n_residual_blocks 6 \\
       --lambda_dice 1.0 \\
       --lambda_perceptual 0.1 \\
       --exp_name my_enhancement_experiment

3. Mit vortrainiertem Segmentierungsmodell:
   
   source .venv/bin/activate && python src/training/train_enhancement.py \\
       --config configs/enhancement_config.yaml \\
       --segmentation_checkpoint path/to/segmentation_model.pth \\
       --freeze_segmentation

4. Training überwachen mit TensorBoard:
   
   tensorboard --logdir experiments/enhancement/
   
   Dann öffnen Sie: http://localhost:6006

╔════════════════════════════════════════════════════════════════════════════╗
║  Ergebnisse werden gespeichert in:                                        ║
║  - experiments/enhancement/<exp_name>/checkpoints/                         ║
║  - experiments/enhancement/<exp_name>/logs/                                ║
║  - experiments/enhancement/<exp_name>/visualizations/                      ║
╚════════════════════════════════════════════════════════════════════════════╝
""")

## 15. Modell speichern und laden

In [ ]:
# Modell speichern
save_path = project_root / 'experiments' / 'enhancement' / 'demo_model.pth'
save_path.parent.mkdir(parents=True, exist_ok=True)

checkpoint = {
    'epoch': 1,
    'enhancement_state_dict': enhancement_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'n_channels': 3,
        'n_output_channels': 3,
        'base_channels': 64,
        'n_residual_blocks': 6
    }
}

torch.save(checkpoint, save_path)
print(f"✓ Model saved to: {save_path}")

# Modell laden
loaded_checkpoint = torch.load(save_path, map_location=device)

# Neues Modell erstellen und laden
loaded_model = EnhancementUNet(**loaded_checkpoint['config'])
loaded_model.load_state_dict(loaded_checkpoint['enhancement_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()

print(f"✓ Model loaded from: {save_path}")
print(f"  Epoch: {loaded_checkpoint['epoch']}")

## 16. Zusammenfassung

### Was haben wir erstellt?

1. **Enhancement U-Net (M_E)**
   - Hybrid-Architektur mit Residual Blocks
   - Leaky ReLU für Low-Contrast-Erhaltung
   - ~47M Parameter (base_channels=64)

2. **Task-Aware Pipeline**
   - End-to-End Enhancement + Segmentation
   - Gefrorenes Segmentierungsmodell
   - Optimierung auf Segmentierungsleistung

3. **Combined Loss Function**
   - Dice Loss (Task-Aware)
   - Perceptual Loss (Qualität)
   - Gewichtete Kombination

### Nächste Schritte:

1. ✓ Vollständiges Training durchführen (~100 Epochen)
2. ✓ Mit CLAHE-Baseline vergleichen
3. ✓ Hyperparameter tunen (λ-Werte, Modellgröße)
4. ✓ Auf Testset evaluieren
5. ✓ Visualisierungen für Paper erstellen

### Erwartete Ergebnisse:

- Dice OD: ~0.90-0.95
- Dice OC: ~0.85-0.92
- Bessere Performance als CLAHE-Baseline
- Task-spezifisch optimierte Enhancements

## 17. Weitere Ressourcen

- **Vollständige Dokumentation**: `ENHANCEMENT_README.md`
- **Trainingsskript**: `src/training/train_enhancement.py`
- **Konfiguration**: `configs/enhancement_config.yaml`
- **Test-Suite**: `src/test_enhancement.py`

### Hilfreiche Befehle:

```bash
# Tests ausführen
source .venv/bin/activate && python src/test_enhancement.py

# Training starten
source .venv/bin/activate && python src/training/train_enhancement.py --config configs/enhancement_config.yaml

# TensorBoard starten
tensorboard --logdir experiments/enhancement/
```